In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, 
                                     BatchNormalization, Add, Activation, GlobalAveragePooling2D)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
import matplotlib.pyplot as plt
import numpy as np
import os
import datetime
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import keras_tuner as kt

In [2]:
# Load and preprocess CIFAR-10 data
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
# Data augmentation pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [4]:
# Data augmentation pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# Define a simple ResNet block
def resnet_block(inputs, filters, kernel_size=3, stride=1):
    x = Conv2D(filters, kernel_size, strides=stride, padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2D(filters, kernel_size, strides=1, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)

    # Skip connection
    if stride != 1 or inputs.shape[-1] != filters:
        shortcut = Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(inputs)
        shortcut = BatchNormalization()(shortcut)
    else:
        shortcut = inputs

    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

In [5]:
# Model building function
def build_model(hp):
    inputs = Input(shape=(32, 32, 3))

    x = data_augmentation(inputs)

    filters = hp.Int('filters_1', 32, 64, step=16)
    x = Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    num_blocks = hp.Int('num_blocks', 2, 3)
    for i in range(num_blocks):
        stride = 2 if i == 0 else 1
        filters = hp.Int(f'filters_block_{i}', filters, filters * 2, step=16)
        x = resnet_block(x, filters, stride=stride)

    x = GlobalAveragePooling2D()(x)

    dense_units = hp.Int('dense_units', 128, 512, step=64)
    x = Dense(dense_units, activation='relu')(x)
    x = Dropout(hp.Float('dropout_rate', 0.2, 0.5, step=0.1))(x)

    outputs = Dense(10, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)

    learning_rate = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

In [ ]:
# Setup Hyperband tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=12,           # lower
    factor=4,                # bigger
    hyperband_iterations=1,  # only 1 round
    directory='my_dir',
    project_name='cifar10_tuning_resnet'
)


# Callbacks
earlystop_cb = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint_cb = ModelCheckpoint('best_cifar10_resnet.h5', save_best_only=True)
reduce_lr_cb = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# TensorBoard for better visualization
log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_cb = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Search for best hyperparameters
tuner.search(x_train, y_train,
             epochs=10,
             validation_split=0.2,
             callbacks=[earlystop_cb],
             verbose=1)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\nBest Hyperparameters:")
for param in best_hps.values.keys():
    print(f" - {param}: {best_hps.get(param)}")

Reloading Tuner from my_dir\cifar10_tuning_resnet\tuner0.json

Search: Running Trial #16

Value             |Best Value So Far |Hyperparameter
32                |48                |filters_1
3                 |2                 |num_blocks
32                |48                |filters_block_0
64                |64                |filters_block_1
192               |320               |dense_units
0.2               |0.3               |dropout_rate
0.001             |0.001             |learning_rate
64                |32                |filters_block_2
7                 |7                 |tuner/epochs
3                 |3                 |tuner/initial_epoch
2                 |2                 |tuner/bracket
1                 |1                 |tuner/round
0004              |0011              |tuner/trial_id




C:\Users\amnaa\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 64 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Epoch 4/7
1200/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 285ms/step - accuracy: 0.5970 - loss: 1.1252

In [ ]:
# Build best model
model = tuner.hypermodel.build(best_hps)

# Save model architecture
with open('model_summary.txt', 'w', encoding='utf-8') as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))

# Train final model
history = model.fit(
    x_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[checkpoint_cb, earlystop_cb, reduce_lr_cb, tensorboard_cb],
    verbose=1
)

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=64, verbose=1)
print(f"\nTest accuracy: {test_acc:.4f}")

In [ ]:
# Plot Accuracy and Loss
plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.show()

In [ ]:
# Confusion Matrix
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
plt.figure(figsize=(10,10))
disp.plot(cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Display some random predictions
num_images = 10
indices = np.random.choice(len(x_test), num_images, replace=False)

plt.figure(figsize=(15,6))
for i, idx in enumerate(indices):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_test[idx])
    pred_label = class_names[y_pred_classes[idx]]
    true_label = class_names[y_true[idx]]
    color = 'green' if pred_label == true_label else 'red'
    plt.title(f"Pred: {pred_label}\nTrue: {true_label}", color=color)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

# Predict
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = y_test.flatten()

# Report
print(classification_report(y_true, y_pred_classes, target_names=class_names))
